In [1]:
# !pip install "langchain<0.2.0" "langsmith<0.1.7" "langchain_openai<0.1.7" "pydantic<2.0"


In [2]:
# !pip show langchain

In [3]:
# !pip show pydantic

In [29]:
from langchain_core.tools import tool #tool 
import random #randomchoose
import requests #connection
from langchain_openai import ChatOpenAI #model
import os #key
from langgraph.prebuilt import create_react_agent #agent
import warnings
warnings.filterwarnings("ignore")

### tool

In [30]:


@tool
def random_movie(genre: str) -> str:
    """Returns a random movie name based on the given genre."""
    
    url = "https://api.npoint.io/1b111baad7e6ea24a1f6"
    genre = genre.title()
    
    response = requests.get(url).json()
    
    if genre not in response:
        return f"No movies found for genre: {genre}"
    
    return random.choice(response[genre])


In [31]:
random_movie('hORROR')

'A Nightmare on Elm Street'

In [32]:
random_movie('comeDy')

'Anchorman'

### Model

In [33]:
# !pip install langchain==0.1.16 langchain-openai==0.0.8 pydantic==2.3


In [34]:
# pip install --upgrade "pydantic<2.5"

In [35]:
# !pip install pydantic==1.10.13


In [36]:


from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url="https://api.together.xyz/v1",
    api_key=os.environ["TOGETHERAI_API_KEY"],
    model="deepseek-ai/DeepSeek-V3",
)

In [37]:
#bind_tools to give model knowledge about these tools
tools = [random_movie]


In [38]:
model_with_tools = model.bind_tools(tools)

In [39]:
tools

[StructuredTool(name='random_movie', description='Returns a random movie name based on the given genre.', args_schema=<class 'langchain_core.utils.pydantic.random_movie'>, func=<function random_movie at 0x00000180AAC90540>)]

In [40]:
# random_movie()

In [41]:
model_with_tools.invoke('hi')

AIMessage(content="Hello! How can I assist you today? If you're looking for a movie recommendation, just let me know the genre you're interested in!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 181, 'total_tokens': 211, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'cached_tokens': 0}, 'model_name': 'deepseek-ai/DeepSeek-V3', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--5ff36bb9-df7f-4f7e-81dc-d960012e3fc4-0', usage_metadata={'input_tokens': 181, 'output_tokens': 30, 'total_tokens': 211, 'input_token_details': {}, 'output_token_details': {}})

In [42]:
model_with_tools.invoke('hi').content

'Hello! How can I assist you today?'

In [45]:
print(model_with_tools.invoke('tell me 5 best places in USA').content)

While I can't directly list the "best" places in the USA (as it depends on personal preferences), here are five popular and highly recommended destinations:

1. **New York City, New York** - Known for its iconic landmarks like Times Square, Central Park, and the Statue of Liberty.
2. **Grand Canyon, Arizona** - A breathtaking natural wonder with stunning views and hiking trails.
3. **San Francisco, California** - Famous for the Golden Gate Bridge, Alcatraz Island, and its vibrant culture.
4. **Orlando, Florida** - Home to world-renowned theme parks like Walt Disney World and Universal Studios.
5. **Washington, D.C.** - The capital of the USA, rich in history, museums, and monuments like the Lincoln Memorial.

Let me know if you'd like more details or recommendations for a specific type of destination!


In [47]:
model_with_tools.invoke('tell me 5 best places in USA').tool_calls

[]

In [48]:
model_with_tools.invoke('any Comedy movie').tool_calls

[{'name': 'random_movie',
  'args': {'genre': 'Comedy'},
  'id': 'call_hccovvn0y89s3gmsegy11869',
  'type': 'tool_call'}]

### model with tool

In [49]:
prompt = '''Act as a helpful assistant who provides general answers to general query and also talk in a general way
            but when   a user asks
            for a movie recommendation based on genre use tool random_movie.

        - random_movie - only use the tool when user asks for movie recommendation.
            
        - check the genre used by user as Comedy , Action , Drama, Horror,Scifi and recommed the user with the movie.

       
        if above mentioned Genre are not present say I DONT KNOW REGARDING THIS GENRE SINCE INFORMATION IS UNAVAILAIBLE and exit.


'''

print(model_with_tools.invoke(prompt).content)
print('--------------')
print(model_with_tools.invoke(prompt).tool_calls)

Got it! I'll be a helpful assistant for general queries and provide movie recommendations only when specifically asked, based on the genres: Comedy, Action, Drama, Horror, or Sci-fi. If the genre isn't one of these, I'll let the user know that the information is unavailable. 

How can I assist you today?
--------------
[]


In [50]:

agent = create_react_agent(model=model, tools=tools,prompt = prompt)

In [51]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]

        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print() 

inputs = {"messages": [("user", "hi")]}

print_stream(agent.stream(inputs, stream_mode="values"))  # Ensure agent is properly defined


================================ Human Message =================================

hi
================================== Ai Message ==================================

Hello! How can I assist you today? Let me know if you have any questions or need recommendations.


In [52]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]
        #print(message)

        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print() 

inputs = {"messages": [("user", "5 best places to visit in USA")]}

print_stream(agent.stream(inputs, stream_mode="values"))  # Ensure agent is properly defined


================================ Human Message =================================

5 best places to visit in USA
================================== Ai Message ==================================

The United States has many amazing places to visit, but here are five popular and diverse destinations that offer unique experiences:

1. **New York City, New York** – Known for its iconic landmarks like Times Square, Central Park, the Statue of Liberty, and Broadway shows.
  
2. **Grand Canyon, Arizona** – A breathtaking natural wonder with stunning views, hiking trails, and outdoor activities.

3. **San Francisco, California** – Famous for the Golden Gate Bridge, Alcatraz Island, and its vibrant cultural scene.

4. **Orlando, Florida** – A dream destination for families, with Walt Disney World, Universal Studios, and other theme parks.

5. **New Orleans, Louisiana** – Rich in culture, music (especially jazz), and delicious cuisine like gumbo and beignets.

Would you like recommendations based 

In [53]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]

        if isinstance(message, tuple):
            print(message)
        else:
            
            message.pretty_print() 

inputs = {"messages": [("user", "ANY DRAMA MOVIE?")]}

print_stream(agent.stream(inputs, stream_mode="values"))  # Ensure agent is properly defined


================================ Human Message =================================

ANY DRAMA MOVIE?
================================== Ai Message ==================================
Tool Calls:
  random_movie (call_hyu8phmur0bmm73ljilr61lg)
 Call ID: call_hyu8phmur0bmm73ljilr61lg
  Args:
    genre: Drama
================================= Tool Message =================================
Name: random_movie

Forrest Gump
================================== Ai Message ==================================

Here's a great drama movie recommendation for you: **"Forrest Gump"**. Enjoy!


In [55]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]

        if isinstance(message, tuple):
            print(message)
        else:
            
            message.pretty_print() 

inputs = {"messages": [("user", "tHriLLER?")]}

print_stream(agent.stream(inputs, stream_mode="values"))  # Ensure agent is properly defined


================================ Human Message =================================

tHriLLER?
================================== Ai Message ==================================

I DON'T KNOW REGARDING THIS GENRE SINCE INFORMATION IS UNAVAILABLE.


**in tool calling in agents do things work like this way.
first there is a user message , if tool calling is not there then LLM will provide output based on previous knowledge. but if tool calling , then llm will call the tool ,tool will return a message to llm and then llm will provid the answer to the user.
m i right?**

Yes, you're absolutely right! Here's a structured breakdown of how **tool calling in agents** works:

---

### **Flow of Tool Calling in Agents**
1. **User sends a message**  
   - Example: `"What is the weather in New York?"`

2. **LLM checks if tool calling is needed**
   - If the model **already knows the answer**, it **responds directly**.  
   - If **tool calling is enabled**, and the query **requires external data**, the LLM **decides to call the tool**.

3. **LLM generates a tool invocation request**
   - It structures a request for the tool (e.g., an API).  
   - Example:  
     ```
     Call weather_api(location="New York")
     ```

4. **Tool processes the request and returns a response**
   - The tool fetches **real-time data**.  
   - Example API response:
     ```json
     {"temperature": "15°C", "condition": "Sunny"}
     ```

5. **LLM interprets the tool's response**
   - The LLM **reads the tool’s output** and **formats it into a user-friendly response**.

6. **Final response is sent to the user**
   - Example:
     `"The current temperature in New York is 15°C, and it's sunny."`

---



---

### **Key Takeaways**
✅ **Without Tool Calling:** LLM answers based on its **training data**.  
✅ **With Tool Calling:** LLM **requests real-time data**, then **processes and responds**.  
✅ **LLM doesn’t directly use the tool output**—it **interprets and reformulates it** before replying.  

Your understanding is spot on! 🚀